In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_hourlyprices_last90days(
  datetime TIMESTAMP,
  prices DOUBLE,
  market_caps DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/bitcoin_hourlyprices_last90days/'

In [0]:
%sql
    CREATE TABLE IF NOT EXISTS silver_5minprices_lastday(
      datetime TIMESTAMP,
      prices DOUBLE,
      market_caps DOUBLE
    )
    USING DELTA
    LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/live/bitcoin_5minprices_lastday/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_4hourlyohlc_last30days(
  datetime TIMESTAMP,
  open DOUBLE,
  high DOUBLE,
  low DOUBLE,
  close DOUBLE
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/bitcoin_4hourlyohlc_last30days/'

In [0]:
%sql
    CREATE TABLE IF NOT EXISTS silver_30minohlc_lastday(
      datetime TIMESTAMP,
      open DOUBLE,
      high DOUBLE,
      low DOUBLE,
      close DOUBLE
    )
    USING DELTA
    LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/live/bitcoin_30minohlc_lastday/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_currency(
  currency_code STRING,
  currency_key STRING,
  currency_name STRING,
  currency_symbol STRING,
  is_active BOOLEAN
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/silver_currency_table/'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver_coin(
  coin_id STRING,
  coin_key STRING,
  founded_year Date,
  ticker_symbol STRING,
  `source` STRING,
  is_active BOOLEAN
)
USING DELTA
LOCATION 'abfss://silver@bitcoindatalake.dfs.core.windows.net/history/silver_coins_table/'

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO silver_hourlyprices_last90days as target
USING(
  with seperated_val_CTE(
  SELECT from_json(market_caps, 'ARRAY<ARRAY<DOUBLE>>') as m_caps,
  from_json(prices, 'ARRAY<ARRAY<DOUBLE>>') as prc,
  from_json(total_volumes, 'ARRAY<ARRAY<DOUBLE>>') as tot_vol
  from bitcoinproject_workspace.default.bronze_chart_last90days
  ),
  zipped_array(
  SELECT
  explode(arrays_zip(m_caps, prc, tot_vol)) as merged_row
  FROM seperated_val_cte
  ),
  extract_cols_CTE
  (
    select
    FROM_UNIXTIME(merged_row.m_caps[0]/1000) as datetime,
    CAST(merged_row.prc[1] as double) as prices,
    CAST(merged_row.m_caps[1] as double) as market_caps,
    CAST(merged_row.tot_vol[1] as double) as total_volumes
    from zipped_array
  )
  select *
  FROM extract_cols_cte
  where datetime <= current_date()
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'incremental_load' THEN
  MERGE INTO silver_5minprices_lastday as target
    USING(
      with seperated_val_CTE(
      SELECT from_json(market_caps, 'ARRAY<ARRAY<DOUBLE>>') as m_caps,
      from_json(prices, 'ARRAY<ARRAY<DOUBLE>>') as prc,
      from_json(total_volumes, 'ARRAY<ARRAY<DOUBLE>>') as tot_vol
      from bronze_chart_lastday
      ),
      zipped_array(
      SELECT
      explode(arrays_zip(m_caps, prc, tot_vol)) as merged_row
      FROM seperated_val_cte
      ),
      extracted_cols_CTE
      (
        select
        FROM_UNIXTIME(merged_row.m_caps[0]/1000) as datetime,
        CAST(merged_row.prc[1] as double) as prices,
        CAST(merged_row.m_caps[1] as double) as market_caps,
        CAST(merged_row.tot_vol[1] as double) as total_volumes
        from zipped_array
      )
      select *
      FROM extracted_cols_cte
      -- where datetime >= current_date()
    ) as source
    on target.datetime = source.datetime
    WHEN MATCHED THEN
    UPDATE SET *
    WHEN NOT MATCHED THEN
    INSERT *;
  END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO silver_4hourlyohlc_last30days as target
USING(
  with exploded_CTE(
  SELECT explode(*) as collec
  FROM bitcoinproject_workspace.default.bronze_ohlc_last30days
),
extracted_cols_CTE
(
  select
  from_unixtime(collec[0]/1000) as datetime,
  CAST(collec[1] as DOUBLE) as open,
  CAST(collec[2] as double) as high,
  CAST(collec[3] as double) as low,
  CAST(collec[4] as double) as close
  FROM exploded_cte  
)
select *
FROM extracted_cols_cte
where datetime <= current_date()
) as source
on target.datetime = source.datetime
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'incremental_load' THEN
    MERGE INTO silver_30minohlc_lastday as target
    USING(
      with exploded_CTE(
      SELECT explode(*) as collec
      FROM bronze_ohlc_lastday
    ),
    extracted_cols_CTE
    (
      select
      from_unixtime(collec[0]/1000) as datetime,
      CAST(collec[1] as DOUBLE) as open,
      CAST(collec[2] as double) as high,
      CAST(collec[3] as double) as low,
      CAST(collec[4] as double) as close
      FROM exploded_cte  
    )
    select *
    FROM extracted_cols_cte
    -- where datetime >= current_date()
    ) as source
    on target.datetime = source.datetime
    WHEN MATCHED THEN
    UPDATE SET *
    WHEN NOT MATCHED THEN
    INSERT *;
  END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO silver_coin as target
USING(
  SELECT *
  FROM bronze_coin
) as source
on target.coin_key = source.coin_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;

In [0]:
%sql
BEGIN
  IF :mode == 'initial_load' THEN
MERGE INTO silver_currency as target
USING(
  SELECT *
  FROM bronze_currency
) as source
on target.currency_key = source.currency_key
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
END IF;
END;